# 08 — Evaluation: All Six Models Side by Side

Loads every model's real `results/metrics/*.json` (produced by notebooks 02–07) and reproduces `results/tables/main_results.md`'s comparison table. Metrics are computed by `pytrec_eval` (`src/biomedical_ir/evaluation.py::evaluate_run`), cross-checked against custom from-scratch implementations in the same module (`tests/test_metrics.py`).

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
import json
import pandas as pd

models = ['tfidf', 'bm25', 'bge', 'medcpt', 'hybrid_rrf', 'hybrid_reranked']
rows = []
for m in models:
    d = json.load(open(f'results/metrics/{m}.json'))
    metrics = d['metrics']
    rows.append({
        'model': m,
        'P@10': metrics['P@10'],
        'Recall@100': metrics['Recall@100'],
        'MAP': metrics['MAP'],
        'MRR@10': metrics['MRR@10'],
        'nDCG@10': metrics['nDCG@10'],
        'latency_ms': d.get('timing', {}).get('latency_ms_per_query'),
    })
df = pd.DataFrame(rows).set_index('model')
df

nDCG@10 progression across the study, best to worst so far: Hybrid+Reranker (0.3731) > BGE (0.3712) > MedCPT (0.3654) > Hybrid RRF (0.3620) > TF-IDF (0.3050) > BM25 (0.2954). See notebook 09 for which of these differences are statistically significant.